# 04 - Gold Business Model

Builds the analytics-ready star schema used by the SQL analytics endpoint and the Direct Lake semantic model:

- `gold_dim_customer`, `gold_dim_product`, `gold_dim_date`
- `gold_fact_sales`
- `gold_sales_summary` (pre-aggregated daily KPIs)

Gold tables are written with **V-Order**, which Fabric disables by default in new workspaces.
V-Order speeds up reads by Direct Lake, the SQL analytics endpoint and Power BI.

In [ ]:
from pyspark.sql import functions as F

def write_gold(df, table_name):
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .option("parquet.vorder.enabled", "true")   # read-optimized layout for Direct Lake
       .saveAsTable(table_name))
    print(f"{table_name}: {spark.table(table_name).count():,} rows")

customers = spark.table("silver_customers")
products = spark.table("silver_products")
sales = spark.table("silver_sales")

# Dimensions
dim_customer = customers.select("customer_id", "customer_name", "segment", "city", "state", "region", "join_date")
dim_product = products.select(
    "product_id", "product_name", "category", "unit_cost", "list_price",
    (F.col("list_price") - F.col("unit_cost")).alias("unit_margin"),
)

# Date dimension covering every order date (needed for Power BI time intelligence)
bounds = sales.agg(F.min("order_date").alias("start"), F.max("order_date").alias("end")).first()
dim_date = (
    spark.range(1)
    .select(F.explode(F.sequence(F.lit(bounds["start"]), F.lit(bounds["end"]))).alias("date"))
    .select(
        "date",
        F.year("date").alias("year"),
        F.quarter("date").alias("quarter"),
        F.month("date").alias("month"),
        F.date_format("date", "MMM").alias("month_name"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
        F.dayofmonth("date").alias("day"),
        F.date_format("date", "EEE").alias("day_name"),
        F.dayofweek("date").isin(1, 7).alias("is_weekend"),
    )
)

# Fact table: one row per order line
fact_sales = sales.select(
    "order_id", "order_date", "customer_id", "product_id",
    "quantity", "unit_price", "discount_pct", "sales_amount", "cost_amount",
    (F.col("sales_amount") - F.col("cost_amount")).alias("gross_profit"),
)

# Pre-aggregated daily summary for fast KPI queries
sales_summary = (
    fact_sales
    .join(dim_customer.select("customer_id", "segment", "region"), "customer_id", "left")
    .join(dim_product.select("product_id", "category"), "product_id", "left")
    .groupBy("order_date", "region", "segment", "category")
    .agg(
        F.sum("sales_amount").alias("total_sales"),
        F.sum("gross_profit").alias("gross_profit"),
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("order_id").alias("orders"),
    )
)

write_gold(dim_customer, "gold_dim_customer")
write_gold(dim_product, "gold_dim_product")
write_gold(dim_date, "gold_dim_date")
write_gold(fact_sales, "gold_fact_sales")
write_gold(sales_summary, "gold_sales_summary")
print("Gold layer created.")